# 1er Livrable

*AUDIN Loric, FAIVRE Elio, PERREY PRATO Joan, RUFFIN Axel*

## 1.Contexte
L’ADEME (Agence de l’Environnement et de la Maîtrise de l’Énergie) souhaiterait ralentir le réchauffement climatique en essayant de diviser par 4 les émissions d'ici 2050 pour la France. Pour cela, elle aimerait optimiser la logistique du transport en limitant au maximum les déplacements et la consommation des véhicules lors des livraisons.

Nous, CesiCDP, devons les aider à atteindre les objectifs et permettre de trouver les trajets optimaux rapidement dans toute la France. Il leur faudrait une solution pour résoudre leur problème d'optimisation ainsi que des études de celles-ci.

## 2. Identification du problème

Pour répondre aux objectifs de l'ADEME, notre modèle ne se contente pas de chercher le chemin le plus court car il doit intégrer les réalités logistiques via deux contraintes majeures :

- Coût ou restriction de passage sur certaines arêtes : Contrairement à un modèle simplifié où toutes les routes sont accessibles, notre modèle intègre des pondérations variables sur les arêtes ($c_{ij}$). Cela nous permet de modéliser des travaux programmés ou des restrictions de gabarit. Certaines arêtes peuvent être totalement interdites (coût infini), forçant l'algorithme à recalculer un détour optimal.

- Fenêtres temporelles (Time Windows) : Chaque ville $i$ est associée à un intervalle de temps $[e_i, l_i]$. Le véhicule doit impérativement arriver avant l'heure limite $l_i$. S'il arrive avant l'heure d'ouverture $e_i$, il doit patienter, ce qui impacte le temps total de la tournée. Cette contrainte est capitale pour modéliser les horaires de livraison en zone urbaine ou les créneaux de réception des clients.

Pour la représentation du réseau routier national sous forme de graphe, nous considérerons que les noeuds sont des villes et les arêtes sont les routes.


## 3.Explicitation des contraintes (Avec équations)

Pour transformer le problème du voyageur de commerce classique(TSP) en un problème avec fenêtres temporelles (TSPTW), nous devons introduire une variable de décision supplémentaire :
- $t_i$ : l'instant auquel le véhicule commence à servir la ville $i$.
### A. Contrainte de Fenêtre Temporelle
Pour chaque sommet $i \in S$, le passage doit s'effectuer dans l'intervalle imparti :$$e_i \le t_i \le l_i$$
- $e_i$ : heure d'ouverture au plus tôt (earliest).
- $l_i$ : heure de fermeture au plus tard (latest).
### B. Contrainte de cohérence chronologique
Si le véhicule se déplace de la ville $i$ vers la ville $j$ ($x_{ij} = 1$), alors l'heure d'arrivée en $j$ doit être supérieure ou égale à l'heure de départ de $i$ augmentée du temps de trajet $d_{ij}$ :$$t_i + d_{ij} \le t_j + M(1 - x_{ij})$$(Où $M$ est une constante très grande permettant d'activer/désactiver la contrainte selon la valeur de $x_{ij}$).
- Si $x_{ij} = 1$ : la contrainte devient $t_i + d_{ij} \le t_j$ (le temps de trajet est respecté).
- Si $x_{ij} = 0$ : la contrainte est trivialement satisfaite grâce à la constante $M$.
### C. Restriction de passage sur les arêtes
Le réseau routier peut présenter des restrictions (travaux, interdictions). Pour modéliser l'impossibilité d'emprunter une route entre $i$ et $j$, on définit un coût $c_{ij}$ prohibitif ou on retire l'arête de l'ensemble des arcs admissibles $A$ :$$c_{ij} = \infty \implies x_{ij} = 0$$Cette restriction force l'algorithme à chercher un chemin alternatif, impactant directement la distance totale et le respect des fenêtres temporelles des villes suivantes.

## 4. Définition mathématique du problème

### Formulation précise

On établit le problème de décision suivant :
Soit un graphe orienté $G(S, A)$ avec : <br>
- $S = \{s_0, s_1, \dots, s_n\}$ est l'ensemble des sommets,<br>
- $A$ l'ensemble des arêtes,<br>
- $c_{ij}(t)$ le coût de la traversée de l'arête $(s_i, s_j)$ à l'instant $t$
- $d_{ij}$ le temps de trajet nécéssaire pour parcourir l'arête $(s_i, s_j)$.
- $[e_i, l_i]$ la fenêtre temporelle de la ville $s_i$, où $e_i$ est l'heure d'arrivée au plus tôt et $l_i$ l'heure au plus tard. ($e$ pour "earliest" et $l$ pour "latest")

### Fonction objectif

Notre but est de trouver un cycle hamiltonien qui traverse chaque sommet une seule fois, tout en minimisant la somme des coûts de traversée

$$ \quad Z = \sum_{(i,j) \in A} c_{ij}(t) \cdot x_{ij}$$

Où $x_{ij}$ est une variable binaire valant 1 si l'arête $(i,j)$ est empruntée, 0 sinon.

## 5. Complexité

Pour démontrer la complexité de notre problème d'optimisation de tournées avec restrictions de passage et perturbations dynamiques, nous allons procéder par étape : définir le problème de décision de base, déterminer le problème de décision du TSP,  comparer avec notre problème TSPTW, et démontrer que le problème est NP-difficile.

### 5.1. Définition du problème de décision
Notre étude s'appuie fondamentalement sur le Problème du Voyageur de Commerce (TSP - Traveling Salesperson Problem). Pour évaluer sa complexité théorique, nous devons d'abord considérer sa version de décision :

- Données : graphe complet pondéré G=(S,A,c) (où S est l'ensemble des sommets, A l'ensemble des arêtes et c la fonction de coût associée) et un réel K >= 0.
- Question : Existe-t-il un cycle hamiltonien dont le coût total est inférieur ou égal à K ?

Il a été formellement démontré par Richard M. Karp en 1972 que le problème du cycle hamiltonien est NP-complet. Le TSP-D étant la réduction polynomiale du problème du cycle hamiltonien, le TSP de décision est classé NP-complet. Par conséquent, le problème d'optimisation consistant à trouver la tournée minimale (le TSP classique) est NP-difficile.

### 5.2 Complexité du problème de décision de base (TSP)
Les étapes de résolution du problème comprennent un parcours des villes. Or il existe (n-1)! parcours existant pour les graphes orientés (et (n-1)!/2 pour non orienté). Par exemple, pour un problème à 69 villes, le nombre de chemins possibles est un nombre à 100 chiffres. La complexité du problème est au minimum exponentielle, et plus précisément O(n!). Le problème ne peut donc pas être résolu de manière polynomiale.

En ce qui concerne l’algorithme de certificat, les étapes sont les suivantes :

- Vérifier que la liste S des sommets est bien une chaîne du graphe de l'instance I : Cette étape se fait en O(n). En effet, il y a au plus n-1 arêtes dans une solution valide. S’il y a plus de n-1 arêtes, on sait que la solution est invalide.
- Vérifier que chaque sommet de I n’y apparaît qu’une fois : Cette étape se fait en O(n). Il suffit d’implémenter un dictionnaire associant à chaque sommet le fait qu’il a été visité pendant le parcours de S.
- Vérifier si S a pour extrémités u et v se fait en O(1).
- Vérifier que u et v sont identiques se fait en O(1).

Chaque étape étant de complexité polynomiale, l’algorithme de vérification est lui-même polynomial. Le problème est donc bien dans NP.

### 5.3 Adaptation à notre cas : TSP Dynamique avec restrictions (DTSP)
Dans le cadre de notre projet pour l'ADEME, le problème n'est pas un TSP statique classique. Nous avons ajouté deux contraintes majeures qui modifient le graphe G :


- Coût ou restriction de passage sur certaines arêtes : Notre graphe ne sera donc pas complet dûe aux routes bloquées. Alors, certaines arêtes n’existeront pas ou possèderont un coût ( travaux, routes bloquées, etc).
Ce qui revient en modèle mathématique à définir un coût Cij = ∞ pour une arètes (i,j) € A afin qu’on ne le prenne pas en compte dans le programme car cela ne serait pas optimisé.
- Routes dynamiques ou perturbations : Le coût ne sera donc plus statique en vue des perturbations et tout cela pendant l'exécution.
La fonction de coût devient dépendante du temps ou de l'état d'avancement : Cij(t)

Notre problème est donc une variante d'optimisation combinatoire dynamique, proche du Dynamic Traveling Salesperson Problem (DTSP) puisque dans notre cas, les conditions de circulation dans le graphe peuvent changer ( arêtes non disponible, ou coût perturbé).

### 5.4 Démonstration de la NP-difficulté de notre problème



1. Le point de départ (L'instance de TSP)

On prend n'importe quelle instance du TSP classique.
Définition : Un graphe complet G = (S, A) avec des coûts statiques Cij pour chaque arête reliant la ville i à la ville j. L'objectif est de trouver la tournée de coût minimum.

2. La transformation en temps polynomial (La création du DTSP)

On doit maintenant transformer cette instance de TSP en une instance de notre problème (le DTSP), et cette traduction doit se faire très rapidement (en temps polynomial).
Comment on fait ? On crée une instance de DTSP où l'on définit la fonction de coût dynamique C’ij(t) et l'état des routes de la façon suivante :
Pour chaque arête, on dit que le coût ne change jamais avec le temps : C’ij(t) = Cij pour tout instant t.
On s'assure qu'aucune route ne peut être bloquée (aucune restriction d'arête).
Temps de la transformation : Cette opération consiste simplement à copier les coûts initiaux et à fixer le paramètre temps à une constante. Cela se fait en parcourant les arêtes, soit un temps de O(n²), ce qui est bien une durée polynomiale.

3. L'équivalence des solutions

Il faut montrer que résoudre cette instance de DTSP résout le TSP classique.
Puisque dans notre instance de DTSP, le temps n'a aucun impact et qu'aucune route ne se bloque, l'algorithme qui cherchera la meilleure tournée pour ce DTSP trouvera exactement la même tournée optimale que s'il cherchait à résoudre le TSP d'origine.

4. La conclusion logique (La NP-difficulté)

On a prouvé que si on avait un algo pour résoudre le DTSP, on pourrait l'utiliser pour résoudre le TSP classique simplement en désactivant les options de temps et de routes bloquées (ce qui prend un temps dérisoire, polynomial).
Puisque le TSP classique est intraitable (NP-difficile), cela signifie que notre problème DTSP est au moins aussi difficile à résoudre.
Conclusion finale : Le DTSP est donc NP-difficile.

## 6. Documentation de l'étude

Note : Référence fondatrice démontrant que le problème du voyageur de commerce de décision appartient bien à la classe NP. 

- CHEHILI, Iyad. Problème du voyageur de commerce. ResearchGate, 2022. [Disponible ici](https://www.researchgate.net/profile/Iyad_Chehili/publication/358600281_Probleme_du_Voyageur_de_Commerce_PVC_Etude_theorique_du_probleme_Etude_Experimentale/links/620afa46634ff774f4ce579a/Probleme-du-Voyageur-de-Commerce-PVC-Etude-theorique-du-probleme-Etude-Experimentale.pdf?origin=publication_detail)



Note : Référence fondatrice démontrant la NP-complétude du problème du voyageur de commerce via la réduction du problème du cycle hamiltonien.

- KARP, Richard M. Reducibility among combinatorial problems In Complexity of computer computations. Boston, MA : Springer, 1972. p. 85-103. [Disponible ici](https://cgi.di.uoa.gr/~sgk/teaching/grad/handouts/karp.pdf)

